In [ ]:
import os

BASE = "/kaggle/input/datasets/filipacalheiros"

DATA_ROOT = f"{BASE}/deepfashion-data"
CODE_ROOT = f"{BASE}/project-code"

print("DATA_ROOT:", DATA_ROOT)
print("CODE_ROOT:", CODE_ROOT)

print("DATA_ROOT exists:", os.path.exists(DATA_ROOT))
print("CODE_ROOT exists:", os.path.exists(CODE_ROOT))

print("DATA files:", os.listdir(DATA_ROOT))
print("CODE files:", os.listdir(CODE_ROOT))
print("Image folders example:", os.listdir(f"{DATA_ROOT}/img")[:5])


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
!pip install -q pandas scikit-learn pillow tqdm


In [ ]:
!python "$CODE_ROOT/src/train.py" \
  --metadata "$DATA_ROOT/metadata.csv" \
  --data-root "$DATA_ROOT" \
  --epochs 10 \
  --batch-size 8 \
  --image-size 128 \
  --max-train-batches 2 \
  --max-val-batches 2 \
  --no-pretrained \
  --output-dir /kaggle/working/debug_checkpoints


In [ ]:
import os
import pandas as pd

metadata_path = f"{DATA_ROOT}/metadata.csv"
df = pd.read_csv(metadata_path)

existing = set()

for dirpath, _, filenames in os.walk(f"{DATA_ROOT}/img"):
    for filename in filenames:
        rel = os.path.relpath(os.path.join(dirpath, filename), DATA_ROOT)
        existing.add(rel.replace("\\", "/"))

exists_mask = df["image_path"].isin(existing)

print("Total rows:", len(df))
print("Existing images:", int(exists_mask.sum()))
print("Missing images:", int((~exists_mask).sum()))

print("Missing examples:")
print(df.loc[~exists_mask, "image_path"].head(10).to_string(index=False))

filtered_path = "/kaggle/working/metadata_filtered.csv"
df.loc[exists_mask].to_csv(filtered_path, index=False)

print("Saved:", filtered_path)


In [ ]:
!python "$CODE_ROOT/src/train.py" \
  --metadata /kaggle/working/metadata_filtered.csv \
  --data-root "$DATA_ROOT" \
  --epochs 10 \
  --batch-size 32 \
  --image-size 224 \
  --max-train-batches 3000 \
  --max-val-batches 500 \
  --lr 1e-4 \
  --output-dir /kaggle/working/checkpoints


In [ ]:
!python "$CODE_ROOT/src/infer.py" \
  --checkpoint /kaggle/working/checkpoints/best_resnet50.pt \
  --data-root "$DATA_ROOT" \
  --image "img/Sheer_Pleated-Front_Blouse/img_00000001.jpg" \
  --top-k 10


In [ ]:
import shutil
from pathlib import Path

src = Path("/kaggle/working/checkpoints/best_resnet50.pt")
dst = Path("/kaggle/working/best_resnet50.pt")

shutil.copy2(src, dst)

print("Saved for download:", dst)
